In [ ]:
!pip install transformers datasets torchmetrics

In [ ]:
import torch
import numpy as np

from datasets import load_dataset, Audio, DatasetDict

from transformers import AutoModel, AutoModelForAudioClassification, AutoFeatureExtractor

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Χρήση συσκευής: {device}")

In [ ]:
model_or = AutoModel.from_pretrained("facebook/data2vec-audio-base-960h").to(device).eval()
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/data2vec-finetuned-IEMOCAP-categorical").to(device).eval()
feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/data2vec-finetuned-IEMOCAP-categorical")

In [ ]:
# Φόρτωση dataset
from datasets import load_dataset, Audio, DatasetDict
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

dataset = load_dataset("AbstractTTS/IEMOCAP", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=16_000))

def merge_and_filter(batch):
    emo = batch["major_emotion"]
    if emo == "excited":
        emo = "happy"
    batch["emotion_4class"] = emo
    return batch

dataset = dataset.map(merge_and_filter)
dataset = dataset.filter(lambda b: b["emotion_4class"] in ["angry", "happy", "neutral", "sad"])

def add_speaker(batch):
    parts = batch["file"].split("_")
    session = parts[0][:5]
    speaker_gender = parts[-1][0]
    batch["speaker_id"] = f"{session}_{speaker_gender}"
    return batch

dataset = dataset.map(add_speaker)

labels = sorted(set(dataset["emotion_4class"]))
num_labels = len(labels)
label2id = {name: i for i, name in enumerate(labels)}
id2label = {i: name for i, name in enumerate(labels)}
print("Κλάσεις:", labels)

splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(splitter.split(X=range(len(dataset)), groups=dataset["speaker_id"]))

dataset = DatasetDict({
    "train": dataset.select(train_idx),
    "test": dataset.select(test_idx),
})

train_speakers = set(dataset["train"]["speaker_id"])
test_speakers = set(dataset["test"]["speaker_id"])
print("Επικάλυψη ομιλητών:", train_speakers & test_speakers)

In [ ]:
MAX_SAMPLES = 6 * 16_000  # 6 δευτερόλεπτα, όπως στο wav2vec2 notebook

def is_valid_audio(batch):
    audio = batch["audio"]["array"][:MAX_SAMPLES]
    if len(audio) == 0:
        return False
    return np.std(audio) >= 1e-4

print("Πριν:", len(dataset["train"]), len(dataset["test"]))
dataset["train"] = dataset["train"].filter(is_valid_audio, load_from_cache_file=False)
dataset["test"] = dataset["test"].filter(is_valid_audio, load_from_cache_file=False)
print("Μετά:", len(dataset["train"]), len(dataset["test"]))

In [ ]:
def extract_embeddings(model, feature_extractor, dataset, num_samples=150, encoder_attr=None):
    probe_model = getattr(model, encoder_attr) if encoder_attr else model
    num_layers = model.config.num_hidden_layers + 1
    layer_embeddings = [[] for _ in range(num_layers)]

    for i in range(num_samples):
        sample = dataset[i]
        audio = sample["audio"]["array"]
        inputs = feature_extractor(
            audio,
            sampling_rate=16000,
            max_length=6*16000,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            outputs = probe_model(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(outputs.hidden_states):
            pooled = h.mean(dim=1)
            layer_embeddings[layer].append(pooled.squeeze().cpu().numpy())

    return [np.array(layer) for layer in layer_embeddings]

In [ ]:
emb_pretrained = extract_embeddings(model_or, feature_extractor, dataset["test"], encoder_attr=None)
emb_finetuned = extract_embeddings(model_ft, feature_extractor, dataset["test"], encoder_attr="data2vec_audio")

del model_or, model_ft
torch.cuda.empty_cache()

In [ ]:
#Συνάρτηση υπολογισμού του CKA
def linear_CKA(X, Y):

    X = X - X.mean(0)
    Y = Y - Y.mean(0)

    hsic = np.linalg.norm(X.T @ Y, ord="fro") ** 2

    var1 = np.linalg.norm(X.T @ X, ord="fro")
    var2 = np.linalg.norm(Y.T @ Y, ord="fro")

    return hsic / (var1 * var2)

In [ ]:
import pandas as pd

num_layers = len(emb_pretrained)
cka_scores = [linear_CKA(emb_pretrained[l], emb_finetuned[l]) for l in range(num_layers)]

cka_results = pd.DataFrame({"layer": range(num_layers), "cka": cka_scores})

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cka_results["layer"], cka_results["cka"], marker="o", color="#065A82")
plt.xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
plt.ylabel("CKA similarity (Pretrained vs Fine-tuned)")
plt.title("Layer-wise CKA: data2vec πριν/μετά το Fine-tuning (IEMOCAP)")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()